# REFERENCE SOLUTION W3D4: Solving the compartment model as an ODE, and surviving a discontinuity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/solutions/week03/w3d4_scipy_solve_ivp_solution.ipynb)

| | |
|---|---|
| Problem ID | `W3D4` |
| Week / day | 3 / 4 |
| Topic | scipy.integrate: solve_ivp, t_eval, checking against an analytic solution |
| Expected time | 20-30 min |

> This is the worked solution. Only open it after you have genuinely tried the problem notebook.


## Scenario

So far every profile you have worked with was a formula you could evaluate
directly. That stops being possible very quickly: add a second compartment,
saturable elimination, a transit absorption chain or a PD effect feeding back
on clearance, and there is no closed form left to write down.

What there always is, is a differential equation. A one compartment model says
something simple about **rates**: the amount in the body falls at a rate
proportional to how much is there.

```
dA/dt = -(CL / V) * A
```

Numerical integration turns that statement into a curve. This problem works
through it on a case you can check, an IV bolus, where the exact answer is
known, so you can confirm the machinery before trusting it somewhere you
cannot check. Then you do an infusion, where the dosing rate switches off part
way through, which is where naive integration quietly goes wrong.

## solve_ivp, and why discontinuities need care

**The shape of the call.** `solve_ivp(fun, t_span, y0, ...)` needs three
things: a function giving the derivative, the interval to integrate over as a
pair, and the starting state as a **sequence**, even with one compartment, so
`[500.0]` rather than `500.0`.

The derivative function takes `(t, y)` in that order, with `y` an array of
states, and returns the derivatives in the same order. For one compartment:

```python
def rhs(t, y, cl, v):
    amount = y[0]
    return [-(cl / v) * amount]
```

Extra parameters are passed through with `args=(cl, v)`.

**`t_eval` controls output, not accuracy.** By default the solver returns the
points it happened to step through, which are chosen for efficiency and are
not evenly spaced. Passing `t_eval=np.linspace(...)` makes it report at the
times you want. It does not make the answer more accurate, it only changes
where it is sampled.

**Accuracy is controlled by `rtol` and `atol`, and the defaults are loose.**
`rtol=1e-3` means the solver is content with roughly three correct digits,
which is fine for a quick sketch and not fine for anything you will compare
against data or integrate to get an AUC. For PK work pass something like
`rtol=1e-8, atol=1e-10`. It costs a few milliseconds and removes solver error
as a thing you have to think about. You will see the difference in part b: on
the defaults the numerical and exact curves differ in the third decimal, and
with tight tolerances they agree to about eight.

The result carries `sol.t`, `sol.y` with one **row per state**, so
`sol.y[0]` is the first compartment, and `sol.success`, which is worth
checking rather than assuming.

**Amounts versus concentrations.** The state is an amount in mg, because that
is what mass balance is written in. Concentration is amount divided by volume.
Mixing the two up is the single most common error in writing PK models, and
it does not announce itself: the curve looks perfectly plausible, just scaled
wrong by a factor of V.

**The discontinuity problem.** An infusion that stops at 2 hours makes the
derivative jump. Adaptive solvers choose their step size by looking at how
smooth the solution has been, so a solver happily taking one hour steps can
step straight over the moment the infusion ended, never sampling the jump, and
return a curve that keeps rising past the end of the infusion.

Nothing errors. `sol.success` is still `True`. The answer is just wrong.

Two fixes. Cap the step size with `max_step` so the solver cannot stride over
the event, which is the blunt and reliable option. Or integrate each phase
separately and hand the end state of one in as the start of the next, which is
exact and is what real PK software does. You will use `max_step` here, but
knowing why it is needed matters more than the argument itself.

## Your task

Three parts, worked one at a time.

| Part | What you work out | Variables to fill in |
|---|---|---|
| a | the ODE, solved numerically after a bolus | `rhs`, `sol`, `conc_num` |
| b | the same thing exactly, and the gap between them | `conc_exact`, `max_abs_err` |
| c | a 2 hour infusion, where the rate switches off | `sol_inf`, `conc_inf`, `cmax_inf`, `t_cmax` |

Below this you will find the **Setup** cell, then a **Toolkit** cell that
demonstrates every move on a tiny made up example, then **Part a**, **Part b**
and **Part c**, each with its own explanation and a cell to write in.

## Setup

Run this cell first. It builds the data with a fixed random seed, so your numbers match the checker and match mine.

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

PROBLEM_ID = "W3D4"
TOPIC = "scipy.integrate: solve_ivp, t_eval, checking against an analytic solution"

# A one compartment model. No noise anywhere: this problem is about the
# machinery being right, so everything is exactly checkable.
CL_L_H = 4.20      # clearance, L/h
V_L = 40.0         # volume of distribution, L
K_EL = CL_L_H / V_L

DOSE_MG = 500.0            # IV bolus
T_GRID = np.linspace(0.0, 24.0, 97)        # every 15 minutes

INF_DOSE_MG = 500.0        # the same dose, given as an infusion
INF_DURATION_H = 2.0
INF_RATE_MG_H = INF_DOSE_MG / INF_DURATION_H
T_GRID_INF = np.linspace(0.0, 12.0, 241)   # every 3 minutes

print("CL = %.2f L/h   V = %.1f L   k = CL/V = %.5f 1/h" % (CL_L_H, V_L, K_EL))
print("bolus    : %.0f mg at t = 0" % DOSE_MG)
print("infusion : %.0f mg over %.0f h, so %.1f mg/h"
      % (INF_DOSE_MG, INF_DURATION_H, INF_RATE_MG_H))
print("")
print("bolus output grid :", T_GRID[:4], "...", T_GRID[-1], " (%d points)" % T_GRID.size)
print("infusion grid     :", T_GRID_INF[:4], "...", T_GRID_INF[-1],
      " (%d points)" % T_GRID_INF.size)


## Toolkit for this problem

| What you want to do | How |
|---|---|
| import the solver | `from scipy.integrate import solve_ivp` |
| write a derivative | `def rhs(t, y, a): return [-a * y[0]]`, t first, y an array |
| solve it | `solve_ivp(rhs, (0, 24), [y0], t_eval=grid, args=(a,))` |
| the times it reported at | `sol.t` |
| the first state over time | `sol.y[0]`, one row per state |
| did it work | `sol.success` |
| stop it striding over a jump | `max_step=0.05` |
| largest absolute difference | `np.max(np.abs(a - b))` |
| where the maximum is | `np.argmax(x)` |

`y0` must be a sequence even with one compartment, so `[500.0]`, not `500.0`.
`sol.y` is always two dimensional, so `sol.y[0]` is how you get the first
compartment out.

Run the cell below, then carry on to **Part a**, which is the next section down.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

# A tiny made up decay, nothing to do with the real parameters.
demo_k = 0.4


def demo_rhs(t, y, k):
    """dy/dt = -k * y. t first, y is an ARRAY, return a sequence."""
    return [-k * y[0]]


print("--- 1. a basic solve ---")
demo_grid = np.linspace(0.0, 5.0, 6)
demo_sol = solve_ivp(demo_rhs, (0.0, 5.0), [100.0], t_eval=demo_grid, args=(demo_k,))
print("success :", demo_sol.success)
print("sol.t   :", demo_sol.t)
print("sol.y shape:", demo_sol.y.shape, " <- one ROW per state")
print("sol.y[0]:", np.round(demo_sol.y[0], 4))
print("exact   :", np.round(100.0 * np.exp(-demo_k * demo_grid), 4))
print("max abs error: %.3e" % float(np.max(np.abs(demo_sol.y[0]
                                                  - 100.0 * np.exp(-demo_k * demo_grid)))))

print("")
print("--- 2. without t_eval you get the solver's own steps ---")
demo_bare = solve_ivp(demo_rhs, (0.0, 5.0), [100.0], args=(demo_k,))
print("times chosen by the solver:", np.round(demo_bare.t, 3))
print("they are not evenly spaced, which is fine for accuracy but awkward to plot")

print("")
print("--- 3. a discontinuous input, and what goes wrong ---")


def demo_inf_rhs(t, y, k, rate, t_off):
    """Zero order input that stops at t_off."""
    r = rate if t < t_off else 0.0
    return [r - k * y[0]]


demo_t = np.linspace(0.0, 6.0, 61)
demo_loose = solve_ivp(demo_inf_rhs, (0.0, 6.0), [0.0], t_eval=demo_t,
                       args=(demo_k, 50.0, 2.0))
demo_tight = solve_ivp(demo_inf_rhs, (0.0, 6.0), [0.0], t_eval=demo_t,
                       args=(demo_k, 50.0, 2.0), max_step=0.05)
print("both report success:", demo_loose.success, demo_tight.success)
print("peak without max_step: %.4f at t = %.2f"
      % (demo_loose.y[0].max(), demo_t[int(np.argmax(demo_loose.y[0]))]))
print("peak with max_step   : %.4f at t = %.2f"
      % (demo_tight.y[0].max(), demo_t[int(np.argmax(demo_tight.y[0]))]))
print("the true peak is at t = 2.00, when the input stops")
print("")
print("the loose solve can keep climbing past the switch off, because its step")
print("size was chosen while the solution was smooth and nothing told it a jump")
print("was coming. No error, no warning, just a wrong curve.")


## Part a. Write the differential equation and solve it

Define `rhs(t, y, cl, v)`. The amount in the body is `y[0]`, and the
derivative is `-(cl / v) * amount`. Return it as a sequence, so a list of one
element, because the solver expects one derivative per state.

Then integrate from 0 to 24 hours starting from the whole bolus in the
compartment, so `y0` is `[DOSE_MG]`. Report at `T_GRID` by passing it as
`t_eval`, and pass the parameters through with `args=(CL_L_H, V_L)`.

Pass `rtol=1e-8, atol=1e-10` as well. The defaults ask for only about three
correct digits, which is not enough for work you will compare against data.

Finally convert to concentration. The state is an **amount** in mg, so divide
by the volume to get mg/L. This is the step people skip, and the resulting
curve looks entirely reasonable while being wrong by a factor of 40.

Fill in `rhs`, the derivative function. `sol`, the object `solve_ivp` returns.
And `conc_num`, the concentration in mg/L at each time in `T_GRID`.

Moves: `def rhs(t, y, cl, v)`, `solve_ivp(fun, (0, 24), [y0], t_eval=...,
args=(...))`, `sol.y[0]`, division.

In [ ]:
# Part a. The one compartment ODE, solved numerically after an IV bolus.

def rhs(t, y, cl, v):
    """dA/dt for one compartment. y[0] is the amount in mg."""
    return [-(cl / v) * y[0]]


sol = solve_ivp(rhs, (0.0, 24.0), [DOSE_MG], t_eval=T_GRID, args=(CL_L_H, V_L),
                rtol=1e-8, atol=1e-10)
conc_num = sol.y[0] / V_L

print("solved ok:", None if sol is None else sol.success)
if conc_num is not None:
    print("conc at 0, 6, 12, 24 h:", np.round(conc_num[[0, 24, 48, 96]], 4))


## Part b. Check it against the answer you already know

An IV bolus into one compartment has an exact solution, so you can measure how
well the integration did rather than hoping.

```
C(t) = (dose / V) * exp(-(CL / V) * t)
```

Compute that on the same `T_GRID`, then take the largest absolute difference
between the two curves across all 97 points.

You should get something tiny, around ten to the minus eight. That number is
the whole point of this part: it is evidence the machinery is assembled
correctly. If instead you get something near five thousandths, your solve ran
on the default tolerances and `rtol` never got passed. When you move to a two compartment model next week, there
is no exact answer to compare against, and the only reason to trust the result
is that you checked the method somewhere you could.

If the error comes out large and roughly proportional to the values
themselves, suspect the volume division rather than the solver.

Fill in `conc_exact`, the analytic concentration at each time in `T_GRID`, and
`max_abs_err`, a single number.

Moves: `np.exp`, `np.abs`, `np.max`.

In [ ]:
# Part b. The exact solution, and how far the numerical one strayed.

conc_exact = (DOSE_MG / V_L) * np.exp(-(CL_L_H / V_L) * T_GRID)
max_abs_err = float(np.max(np.abs(conc_num - conc_exact)))

print("exact at 0, 6, 12, 24 h:",
      None if conc_exact is None else np.round(conc_exact[[0, 24, 48, 96]], 4))
print("max absolute error:", max_abs_err)


## Part c. A 2 hour infusion, where the rate switches off

Same compartment, different input. Instead of arriving instantly, the drug
goes in at a constant `INF_RATE_MG_H` for the first `INF_DURATION_H` hours and
then stops. The compartment starts empty.

```
dA/dt = rate(t) - (CL / V) * A,   where rate(t) = R for t < 2, and 0 after
```

Write that as `rhs_inf(t, y, cl, v, rate, t_off)`, using an ordinary `if` on
`t`, which is a single number here rather than an array, so a plain `if` is
fine.

Integrate from 0 to 12 hours from an empty compartment, reporting at
`T_GRID_INF`, with the same tight tolerances, and **pass `max_step=0.05`**.

A word on why, because it is more interesting than it looks. With tolerances
this tight the solver is already taking small steps, so on this particular
problem it would probably get the right answer without `max_step`. You pass it
anyway, because "probably" depends on a step size you do not control: the
solver picks it by looking at how smooth the solution has been, and a
discontinuity is precisely the thing that history cannot warn it about. The
toolkit cell shows the failure on looser tolerances, where it bites hard.
Relying on tight tolerances to paper over a discontinuity is a bet that
happens to be winning here.

The peak must land exactly at the end of the infusion, because that is the
last moment drug is still arriving faster than it is leaving.

Fill in `sol_inf`, the solver result. `conc_inf`, the concentration in mg/L at
each time. `cmax_inf`, the highest concentration reached. And `t_cmax`, the
time at which it occurs.

Moves: `def rhs_inf(t, y, cl, v, rate, t_off)`, an `if` on `t`,
`solve_ivp(..., max_step=0.05)`, `np.max`, `np.argmax`.

In [ ]:
# Part c. A 2 hour infusion into the same compartment.

def rhs_inf(t, y, cl, v, rate, t_off):
    """dA/dt with a zero order input that stops at t_off."""
    r = rate if t < t_off else 0.0
    return [r - (cl / v) * y[0]]


sol_inf = solve_ivp(rhs_inf, (0.0, 12.0), [0.0], t_eval=T_GRID_INF,
                    args=(CL_L_H, V_L, INF_RATE_MG_H, INF_DURATION_H),
                    max_step=0.05, rtol=1e-8, atol=1e-10)
conc_inf = sol_inf.y[0] / V_L
cmax_inf = float(np.max(conc_inf))
t_cmax = float(sol_inf.t[int(np.argmax(conc_inf))])

print("solved ok:", None if sol_inf is None else sol_inf.success)
print("Cmax = %s mg/L at t = %s h" % (cmax_inf, t_cmax))


## Check your work

Run this cell. It prints PASS, FAIL or TODO for each part. Do not edit it.

In [ ]:
# Grading helpers. You do not need to read or edit this cell.
import base64 as _b64, json as _json, datetime as _dt, textwrap as _tw
import numpy as _np
import pandas as _pd

# Expected answers, encoded so that you do not read them by accident.
_EXP = _json.loads(_b64.b64decode("eyJhbW91bnQiOiBbNTAwLjAsIDQ4Ny4wNDU3NjgxNDA4OTA3NCwgNDc0LjQyNzE2MDUyNzkwMDYsIDQ2Mi4xMzU0ODE2NTI0MjYxNywgNDUwLjE2MjI2MTI5MzEzMjgsIDQzOC40OTkyNDg2NzkxMDg1LCA0MjcuMTM4NDA2ODA0MjM5NywgNDE2LjA3MTkwNjg4ODg5NDQzLCA0MDUuMjkyMTIyOTg1MDkzNTYsIDM5NC43OTE2MjY3MjE0NTQ0NiwgMzg0LjU2MzE4MjE4NDI4NTI3LCAzNzQuNTk5NzQwOTMxMzAxMSwgMzY0Ljg5NDQzNzEzNDUyODQsIDM1NS40NDA1ODI4NDkwNDg2LCAzNDYuMjMxNjYzNDA0MzIxNywgMzM3LjI2MTMzMjkxNDkxMjQsIDMyOC41MjM0MDk5MDc1Mjg0LCAzMjAuMDExODczMDYxMzUzNzUsIDMxMS43MjA4NTcwNTg3NDQ1LCAzMDMuNjQ0NjQ4NTQzNDI2MSwgMjk1Ljc3NzY4MjE4MzQwNzUzLCAyODguMTE0NTM2ODM1ODk5OTcsIDI4MC42NDk5MzE4MTE1OTU3NiwgMjczLjM3ODcyMzIzNTczNDUsIDI2Ni4yOTU5MDA1MDM0NDg1NCwgMjU5LjM5NjU4MjgyNjk0NDY3LCAyNTIuNjc2MDE1ODcyMTQyOSwgMjQ2LjEyOTU2ODQ4MjQ1NTQ4LCAyMzkuNzUyNzI5NDg3NDQ3MDMsIDIzMy41NDExMDQ1OTQxNzc2NSwgMjI3LjQ5MDQxMzM1OTA4Njc0LCAyMjEuNTk2NDg2MjM4MzMwMzMsIDIxNS44NTUyNjE3MTQ1Mzk4NCwgMjEwLjI2Mjc4MzQ5ODAyMjEyLCAyMDQuODE1MTk3ODAwNDcxOTYsIDE5OS41MDg3NTA2NzkzMTg3MiwgMTk0LjMzOTc4NTQ1MDg3NjUsIDE4OS4zMDQ3NDAxNzA1MTYwOCwgMTg0LjQwMDE0NTE3ODEyMTUzLCAxNzkuNjIyNjIwNzA3MTM5OSwgMTc0Ljk2ODg3NDU1NTU3NzY2LCAxNzAuNDM1Njk5ODE3MzM2OTQsIDE2Ni4wMTk5NzI2NzIzMzAzLCAxNjEuNzE4NjUwMjMzODQ5NiwgMTU3LjUyODc2ODQ1MTcwNjY3LCAxNTMuNDQ3NDQwMDY5Njk5OTUsIDE0OS40NzE4NTI2MzYwMDA2NSwgMTQ1LjU5OTI2NjU2NTA4NTksIDE0MS44MjcwMTMyNDk4ODUxNCwgMTM4LjE1MjQ5MzIyMjgzNzIyLCAxMzQuNTczMTc0MzY0NTkxOSwgMTMxLjA4NjU5MDE1OTEyMTQsIDEyNy42OTAzMzc5OTQwMzg4LCAxMjQuMzgyMDc3NTA0OTUzMjMsIDEyMS4xNTk1Mjg5NjI3MTk0OSwgMTE4LjAyMDQ3MTcwMjQ3MjQzLCAxMTQuOTYyNzQyNTkzMzYxOSwgMTExLjk4NDIzNDU0NzkzNDksIDEwOS4wODI4OTUwNzAxMzcyNSwgMTA2LjI1NjcyNDg0MDkzNDM3LCAxMDMuNTAzNzc2MzQwNTc2MywgMTAwLjgyMjE1MjUwNjU1Nzg3LCA5OC4yMTAwMDU0MjYzNDkwMywgOTUuNjY1NTM1MDYzOTk0NDMsIDkzLjE4Njk4ODAxOTcwNDk3LCA5MC43NzI2NTYzMjE1ODYzNywgODguNDIwODc2MjQ4NjcyMjQsIDg2LjEzMDAyNzE4NDQ1MDQzLCA4My44OTg1MzA1MDAwOTI5MywgODEuNzI0ODQ4NDY2NjE5NDIsIDc5LjYwNzQ4MzE5NTI0NTExLCA3Ny41NDQ5NzU2MDUxODI0LCA3NS41MzU5MDQ0MTgxODU0MSwgNzMuNTc4ODg1MTc5MTQ0MDIsIDcxLjY3MjU2OTMwMjA1MzIyLCA2OS44MTU2NDMxNDA2OTk0NiwgNjguMDA2ODI3MDgzNDI0NTgsIDY2LjI0NDg3NDY3MTM0MjUsIDY0LjUyODU3MTczOTQwMjA4LCA2Mi44NTY3MzU1Nzk3MDMzNSwgNjEuMjI4MjE0MTI2NDkwOTU0LCA1OS42NDE4ODUxNjIyNjM0MzUsIDU4LjA5NjY1NTU0NDQ1MDc4LCA1Ni41OTE0NjA0NTIxMjc1MzUsIDU1LjEyNTI2MjY1MjI0MjYwNCwgNTMuNjk3MDUxNzg0ODU5NzIsIDUyLjMwNTg0MzY2NjkxNjM3LCA1MC45NTA2Nzk2MTQwMjEyNTQsIDQ5LjYzMDYyNTc3OTgyMjgzNiwgNDguMzQ0NzcyNTEyNDkzODA1LCA0Ny4wOTIyMzM3Mjc4ODgzMywgNDUuODcyMTQ2Mjk4OTM5NDYsIDQ0LjY4MzY2OTQ2MDg3NjU5NCwgNDMuNTI1OTg0MjMxODUyNjA0LCA0Mi4zOTgyOTI4NDg1ODE4OSwgNDEuMjk5ODE4MjE2NiwgNDAuMjI5ODAzMzc0NzY2MTldLCAiYW1vdW50X2luZiI6IFswLjAsIDEyLjQ2NzI0NDg0NjU4Nzg0NiwgMjQuODY5MjA4MTcxNjY4NjQsIDM3LjIwNjIzMTgwNTE0MTE2NSwgNDkuNDc4NjU1Nzg3MDAxMDQsIDYxLjY4NjgxODM3NjcxMDc5LCA3My44MzEwNTYwNjI1MjQ1OCwgODUuOTExNzAzNTcwNzYyNjQsIDk3LjkyOTA5Mzg3NTAzNjg3LCAxMDkuODgzNTU4MjA1NDI3OTMsIDEyMS43NzU0MjYwNTc2MTYwNCwgMTMzLjYwNTAyNTIwMTk2MTUyLCAxNDUuMzcyNjgxNjkyNTM5OTQsIDE1Ny4wNzg3MTk4NzYxMjgyLCAxNjguNzIzNDYyNDAxMTQ0NjIsIDE4MC4zMDcyMzAyMjY1NDIyMywgMTkxLjgzMDM0MjYzMDY1NTU0LCAyMDMuMjkzMTE3MjE5OTk5MTcsIDIxNC42OTU4Njk5MzgwMjM5MiwgMjI2LjAzODkxNTA3MzgyMjg2LCAyMzcuMzIyNTY1MjcwNzk2MTMsIDI0OC41NDcxMzE1MzUyNjY1NCwgMjU5LjcxMjkyMzI0NTA1MjEsIDI3MC44MjAyNDgxNTc5OTM4LCAyODEuODY5NDEyNDIwNDM3OSwgMjkyLjg2MDcyMDU3NTY3MzgsIDMwMy43OTQ0NzU1NzIzMjg0LCAzMTQuNjcwOTc4NzcyNzE2LCAzMjUuNDkwNTI5OTYxMTQ0MiwgMzM2LjI1MzQyNzM1MjE3ODMzLCAzNDYuOTU5OTY3NTk4ODU4NSwgMzU3LjYxMDQ0NTgwMDg3ODIsIDM2OC4yMDUxNTU1MTI3MTczLCAzNzguNzQ0Mzg4NzUxNzMzLCAzODkuMjI4NDM2MDA2MjA4MywgMzk5LjY1NzU4NjI0MzM1OTgsIDQxMC4wMzIxMjY5MTczMDA4NCwgNDIwLjM1MjM0Mzk3Njk2NTcsIDQzMC42MTg1MjE4NzM5OTA2MywgNDQwLjgzMDk0MzU3MDU1MzM2LCA0NTAuOTg5ODkwNTQ3MTczNTYsIDQ0OC42MjgzOTc5NjM4ODI5LCA0NDYuMjc5MjcwNzI5MjEyNiwgNDQzLjk0MjQ0NDA5NTE5NDczLCA0NDEuNjE3ODUzNjUyODk3MiwgNDM5LjMwNTQzNTMzMDY0ODgsIDQzNy4wMDUxMjUzOTIyNzMxLCA0MzQuNzE2ODYwNDM1MzMxOCwgNDMyLjQ0MDU3NzM4OTM3NzE1LCA0MzAuMTc2MjEzNTE0MjEzNiwgNDI3LjkyMzcwNjM5ODE2ODUsIDQyNS42ODI5OTM5NTYzNzE5LCA0MjMuNDU0MDE0NDI5MDQ1MiwgNDIxLjIzNjcwNjM3OTc5OSwgNDE5LjAzMTAwODY5MzkzOTk1LCA0MTYuODM2ODYwNTc2Nzg1OTcsIDQxNC42NTQyMDE1NTE5OTA2LCA0MTIuNDgyOTcxNDU5ODc2MywgNDEwLjMyMzExMDQ1NTc3NjE1LCA0MDguMTc0NTU5MDA4Mzg0NTYsIDQwNi4wMzcyNTc4OTgxMTYyLCA0MDMuOTExMTQ4MjE1NDc0LCA0MDEuNzk2MTcxMzU5NDI1MSwgMzk5LjY5MjI2OTAzNTc4NjA3LCAzOTcuNTk5MzgzMjU1NjE2LCAzOTUuNTE3NDU2MzMzNjE4LCAzOTMuNDQ2NDMwODg2NTQ5NCwgMzkxLjM4NjI0OTgzMTY0MDI3LCAzODkuMzM2ODU2Mzg1MDIwMDQsIDM4Ny4yOTgxOTQwNjAxNTE4LCAzODUuMjcwMjA2NjY2Mjc2MzUsIDM4My4yNTI4MzgzMDY4NjI2NiwgMzgxLjI0NjAzMzM3ODA2NzcsIDM3OS4yNDk3MzY1NjcyMDM0NiwgMzc3LjI2Mzg5Mjg1MTIxMjg2LCAzNzUuMjg4NDQ3NDk1MTUyNiwgMzczLjMyMzM0NjA1MDY4NTEzLCAzNzEuMzY4NTM0MzU0NTc3MywgMzY5LjQyMzk1ODUyNzIwODEzLCAzNjcuNDg5NTY0OTcxMDgzMywgMzY1LjU2NTMwMDM2OTM1NzksIDM2My42NTExMTE2ODQzNjcsIDM2MS43NDY5NDYxNTYxNjM3NCwgMzU5Ljg1Mjc1MTMwMTA2NTQsIDM1Ny45Njg0NzQ5MTAyMDYwNiwgMzU2LjA5NDA2NTA0ODA5ODc0LCAzNTQuMjI5NDcwMDUxMjAyNywgMzUyLjM3NDYzODUyNjUwMDQ2LCAzNTAuNTI5NTE5MzUwMDgwNiwgMzQ4LjY5NDA2MTY2NTcyOTA3LCAzNDYuODY4MjE0ODgzNTI3MSwgMzQ1LjA1MTkyODY3ODQ1NzMzLCAzNDMuMjQ1MTUyOTg5MDE2MiwgMzQxLjQ0NzgzODAxNTgzNDQ2LCAzMzkuNjU5OTM0MjIwMzA0MywgMzM3Ljg4MTM5MjMyMzIxNDEsIDMzNi4xMTIxNjMzMDMzOTAzLCAzMzQuMzUyMTk4Mzk2MzQ2LCAzMzIuNjAxNDQ5MDkyOTM3MiwgMzMwLjg1OTg2NzEzODAyNTA0LCAzMjkuMTI3NDA0NTI5MTQ3LCAzMjcuNDA0MDEzNTE1MTkyNDUsIDMyNS42ODk2NDY1OTUwODc2NSwgMzIzLjk4NDI1NjUxNjQ4NTgsIDMyMi4yODc3OTYyNzQ0NjQ5NCwgMzIwLjYwMDIxOTExMDIzMjA1LCAzMTguOTIxNDc4NTA5ODM0OTcsIDMxNy4yNTE1MjgyMDI4Nzk0NiwgMzE1LjU5MDMyMjE2MTI1NDQ1LCAzMTMuOTM3ODE0NTk3ODYzMzQsIDMxMi4yOTM5NTk5NjUzNjE3NSwgMzEwLjY1ODcxMjk1NDkwMjMsIDMwOS4wMzIwMjg0OTQ4ODU3NiwgMzA3LjQxMzg2MTc0OTcxODcsIDMwNS44MDQxNjgxMTg1Nzc4LCAzMDQuMjAyOTAzMjM0MTgwNCwgMzAyLjYxMDAyMjk2MTU2MTc1LCAzMDEuMDI1NDgzMzk2ODU4NiwgMjk5LjQ0OTI0MDg2NjA5ODcsIDI5Ny44ODEyNTE5MjM5OTc2NCwgMjk2LjMyMTQ3MzM1Mjc2MDksIDI5NC43Njk4NjIxNjA4OTI5LCAyOTMuMjI2Mzc1NTgyMDExOSwgMjkxLjY5MDk3MTA3MzY3MTM0LCAyOTAuMTYzNjA2MzE2MTg3MjcsIDI4OC42NDQyMzkyMTE0NzE4NywgMjg3LjEzMjgyNzg4MTg3MzE0LCAyODUuNjI5MzMwNjY5MDIwNiwgMjg0LjEzMzcwNjEzMjY3NzIsIDI4Mi42NDU5MTMwNDk1OTY5LCAyODEuMTY1OTEwNDEyMzg4NjYsIDI3OS42OTM2NTc0MjgzODYxMywgMjc4LjIyOTExMzUxODUyMzI2LCAyNzYuNzcyMjM4MzE2MjE1NzUsIDI3NS4zMjI5OTE2NjYyNDg2NCwgMjczLjg4MTMzMzYyMzY2OTQzLCAyNzIuNDQ3MjI0NDUyNjg3LCAyNzEuMDIwNjI0NjI1NTc2NSwgMjY5LjYwMTQ5NDgyMTU5LCAyNjguMTg5Nzk1OTI1ODcyMywgMjY2Ljc4NTQ4OTAyODM4MzMsIDI2NS4zODg1MzU0MjI4MjUyLCAyNjMuOTk4ODk2NjA1NTc1ODUsIDI2Mi42MTY1MzQyNzQ2MjczLCAyNjEuMjQxNDEwMzI4NTMwNDMsIDI1OS44NzM0ODY4NjUzNDQyNiwgMjU4LjUxMjcyNjE4MTU5MTgsIDI1Ny4xNTkwOTA3NzEyMjA2LCAyNTUuODEyNTQzMzI0NTY4OSwgMjU0LjQ3MzA0NjcyNzMzNzQ4LCAyNTMuMTQwNTY0MDU5NTY2NiwgMjUxLjgxNTA1ODU5NDYxODI4LCAyNTAuNDk2NDkzNzk4MTY0MzUsIDI0OS4xODQ4MzMzMjcxNzkxLCAyNDcuODgwMDQxMDI4OTM3NzQsIDI0Ni41ODIwODA5NDAwMTk5NSwgMjQ1LjI5MDkxNzI4NTMxODU2LCAyNDQuMDA2NTE0NDc3MDUzNjgsIDI0Mi43Mjg4MzcxMTM3OTE1NCwgMjQxLjQ1Nzg0OTk3OTQ2OTAyLCAyNDAuMTkzNTE4MDQyNDIyNTcsIDIzOC45MzU4MDY0NTQ0MjMzNiwgMjM3LjY4NDY4MDU0OTcxNTk1LCAyMzYuNDQwMTA1ODQ0MDYzNSwgMjM1LjIwMjA0ODAzMzc5Njg0LCAyMzMuOTcwNDcyOTk0ODY5MjUsIDIzMi43NDUzNDY3ODE5MTU2NiwgMjMxLjUyNjYzNTYyNzMxNzMyLCAyMzAuMzE0MzA1OTQwMjcwODMsIDIyOS4xMDgzMjQzMDU4NjI1LCAyMjcuOTA4NjU3NDg0MTQ3MjMsIDIyNi43MTUyNzI0MDkyMzIxNCwgMjI1LjUyODEzNjE4ODM2NTc4LCAyMjQuMzQ3MjE2MTAxMDMwNjgsIDIyMy4xNzI0Nzk1OTgwNDIzLCAyMjIuMDAzODk0MzAwNjUxMzMsIDIyMC44NDE0Mjc5OTk2NTE1NiwgMjE5LjY4NTA0ODY1NDQ5MjA1LCAyMTguNTM0NzI0MzkyMzkzOCwgMjE3LjM5MDQyMzUwNzQ3MTUyLCAyMTYuMjUyMTE0NDU5ODU5NjgsIDIxNS4xMTk3NjU4NzQ4NDMwNCwgMjEzLjk5MzM0NjU0MTk5MjA4LCAyMTIuODcyODI1NDE0MzAyNTgsIDIxMS43NTgxNzE2MDczNDAwMiwgMjEwLjY0OTM1NDM5ODM4ODM3LCAyMDkuNTQ2MzQzMjI1NjAzMDQsIDIwOC40NDkxMDc2ODcxNjg3MiwgMjA3LjM1NzYxNzU0MDQ2MTUsIDIwNi4yNzE4NDI3MDEyMTUsIDIwNS4xOTE3NTMyNDI2OTE2LCAyMDQuMTE3MzE5Mzk0ODU3MTMsIDIwMy4wNDg1MTE1NDM1NjA3LCAyMDEuOTg1MzAwMjI5NzE4MiwgMjAwLjkyNzY1NjE0ODUwMDUsIDE5OS44NzU1NTAxNDg1MjU3LCAxOTguODI4OTUzMjMxMDU1NDQsIDE5Ny43ODc4MzY1NDkxOTYwNywgMTk2Ljc1MjE3MTQwNzEwMzEsIDE5NS43MjE5MjkyNTkxOTAzOCwgMTk0LjY5NzA4MTcwOTM0MzY1LCAxOTMuNjc3NjAwNTEwMTM3MywgMTkyLjY2MzQ1NzU2MjA1NjI3LCAxOTEuNjU0NjI0OTEyNzIxNDMsIDE5MC42NTEwNzQ3NTYxMTg5MywgMTg5LjY1Mjc3OTQzMTgzNDEyLCAxODguNjU5NzExNDI0Mjg4OCwgMTg3LjY3MTg0MzM2MTk4MzI0LCAxODYuNjg5MTQ4MDE2NzQxNDYsIDE4NS43MTE1OTgzMDI5NjA2NSwgMTg0LjczOTE2NzI3Njg2NTEyLCAxODMuNzcxODI4MTM1NzYzMDMsIDE4Mi44MDk1NTQyMTczMDgxLCAxODEuODUyMzE4OTk4NzY0NTYsIDE4MC45MDAwOTYwOTYyNzYwMywgMTc5Ljk1Mjg1OTI2NDEzODYsIDE3OS4wMTA1ODIzOTQwNzY5NSwgMTc4LjA3MzIzOTUxNDUyNTI4LCAxNzcuMTQwODA0Nzg5OTExMiwgMTc2LjIxMzI1MjUxOTk0MzQsIDE3NS4yOTA1NTcxMzg5MDM5OCwgMTc0LjM3MjY5MzIxNDk0Mjg0LCAxNzMuNDU5NjM1NDQ5Mzc3NTcsIDE3Mi41NTEzNTg2NzU5OTU3NCwgMTcxLjY0NzgzNzg2MDM2MTIyLCAxNzAuNzQ5MDQ4MDk5MTI0NCwgMTY5Ljg1NDk2NDYxOTMzNTQ4LCAxNjguOTY1NTYyNzc3NzYxOTYsIDE2OC4wODA4MTgwNjAyMDk0LCAxNjcuMjAwNzA2MDgwODQ1NCwgMTY2LjMyNTIwMjU4MTUyNzksIDE2NS40NTQyODM0MzExMzYxLCAxNjQuNTg3OTI0NjI0OTA1OTUsIDE2My43MjYxMDIyODM3Njc5NywgMTYyLjg2ODc5MjY1MzY4OTI3LCAxNjIuMDE1OTcyMTA1MDE5MDIsIDE2MS4xNjc2MTcxMzE4MzY2NiwgMTYwLjMyMzcwNDM1MTMwNDcsIDE1OS40ODQyMTA1MDMwMjM2NiwgMTU4LjY0OTExMjQ0ODM5MTEyLCAxNTcuODE4Mzg3MTY5OTY0MjRdLCAiY21heF9hbW91bnQiOiA0NTAuOTg5ODkwNTQ3MTczNTYsICJjbWF4X2luZiI6IDExLjI3NDc0NzI2MzY3OTM0LCAiY29uY19leGFjdCI6IFsxMi41LCAxMi4xNzYxNDQyMDM1MjIyNjgsIDExLjg2MDY3OTAxMzE5NzUxNSwgMTEuNTUzMzg3MDQxMzEwNjUzLCAxMS4yNTQwNTY1MzIzMjgzMiwgMTAuOTYyNDgxMjE2OTc3NzEzLCAxMC42Nzg0NjAxNzAxMDU5OTMsIDEwLjQwMTc5NzY3MjIyMjM2MiwgMTAuMTMyMzAzMDc0NjI3MzQsIDkuODY5NzkwNjY4MDM2MzYyLCA5LjYxNDA3OTU1NDYwNzEzMiwgOS4zNjQ5OTM1MjMyODI1MjcsIDkuMTIyMzYwOTI4MzYzMjEsIDguODg2MDE0NTcxMjI2MjE2LCA4LjY1NTc5MTU4NTEwODA0MywgOC40MzE1MzMzMjI4NzI4MSwgOC4yMTMwODUyNDc2ODgyMDksIDguMDAwMjk2ODI2NTMzODQ1LCA3Ljc5MzAyMTQyNjQ2ODYxMywgNy41OTExMTYyMTM1ODU2NTMsIDcuMzk0NDQyMDU0NTg1MTg4NiwgNy4yMDI4NjM0MjA4OTc0OTksIDcuMDE2MjQ4Mjk1Mjg5ODk0LCA2LjgzNDQ2ODA4MDg5MzM2MywgNi42NTczOTc1MTI1ODYyMTMsIDYuNDg0OTE0NTcwNjczNjE3LCA2LjMxNjkwMDM5NjgwMzU3MywgNi4xNTMyMzkyMTIwNjEzODcsIDUuOTkzODE4MjM3MTg2MTc2LCA1LjgzODUyNzYxNDg1NDQ0MSwgNS42ODcyNjAzMzM5NzcxNjksIDUuNTM5OTEyMTU1OTU4MjU4LCA1LjM5NjM4MTU0Mjg2MzQ5NiwgNS4yNTY1Njk1ODc0NTA1NTMsIDUuMTIwMzc5OTQ1MDExNzk5LCA0Ljk4NzcxODc2Njk4Mjk2OCwgNC44NTg0OTQ2MzYyNzE5MTIsIDQuNzMyNjE4NTA0MjYyOTAyLCA0LjYxMDAwMzYyOTQ1MzAzOCwgNC40OTA1NjU1MTc2Nzg0OTc1LCA0LjM3NDIyMTg2Mzg4OTQ0MSwgNC4yNjA4OTI0OTU0MzM0MjMsIDQuMTUwNDk5MzE2ODA4MjU4LCA0LjA0Mjk2NjI1NTg0NjI0LCAzLjkzODIxOTIxMTI5MjY2NjYsIDMuODM2MTg2MDAxNzQyNDk5LCAzLjczNjc5NjMxNTkwMDAxNjQsIDMuNjM5OTgxNjY0MTI3MTQ4LCAzLjU0NTY3NTMzMTI0NzEyODUsIDMuNDUzODEyMzMwNTcwOTMwNywgMy4zNjQzMjkzNTkxMTQ3OTc2LCAzLjI3NzE2NDc1Mzk3ODAzNTUsIDMuMTkyMjU4NDQ5ODUwOTcsIDMuMTA5NTUxOTM3NjIzODMwOCwgMy4wMjg5ODgyMjQwNjc5ODczLCAyLjk1MDUxMTc5MjU2MTgxMSwgMi44NzQwNjg1NjQ4MzQwNDc0LCAyLjc5OTYwNTg2MzY5ODM3MjUsIDIuNzI3MDcyMzc2NzUzNDMxMywgMi42NTY0MTgxMjEwMjMzNTkzLCAyLjU4NzU5NDQwODUxNDQwNzQsIDIuNTIwNTUzODEyNjYzOTQ3LCAyLjQ1NTI1MDEzNTY1ODcyNTUsIDIuMzkxNjM4Mzc2NTk5ODYwNiwgMi4zMjk2NzQ3MDA0OTI2MjQsIDIuMjY5MzE2NDA4MDM5NjU5MywgMi4yMTA1MjE5MDYyMTY4MDYsIDIuMTUzMjUwNjc5NjExMjYwNSwgMi4wOTc0NjMyNjI1MDIzMjMzLCAyLjA0MzEyMTIxMTY2NTQ4NTMsIDEuOTkwMTg3MDc5ODgxMTI3OSwgMS45Mzg2MjQzOTAxMjk1NjAxLCAxLjg4ODM5NzYxMDQ1NDYzNTQsIDEuODM5NDcyMTI5NDc4NjAwNywgMS43OTE4MTQyMzI1NTEzMzA0LCAxLjc0NTM5MTA3ODUxNzQ4NjYsIDEuNzAwMTcwNjc3MDg1NjE0NCwgMS42NTYxMjE4NjY3ODM1NjI3LCAxLjYxMzIxNDI5MzQ4NTA1MjIsIDEuNTcxNDE4Mzg5NDkyNTgzNywgMS41MzA3MDUzNTMxNjIyNzM5LCAxLjQ5MTA0NzEyOTA1NjU4NTksIDEuNDUyNDE2Mzg4NjExMjY5NSwgMS40MTQ3ODY1MTEzMDMxODgzLCAxLjM3ODEzMTU2NjMwNjA2NTEsIDEuMzQyNDI2Mjk0NjIxNDkyOSwgMS4zMDc2NDYwOTE2NzI5MDkxLCAxLjI3Mzc2Njk5MDM1MDUzMTMsIDEuMjQwNzY1NjQ0NDk1NTcwOCwgMS4yMDg2MTkzMTI4MTIzNDUsIDEuMTc3MzA1ODQzMTk3MjA4LCAxLjE0NjgwMzY1NzQ3MzQ4NjYsIDEuMTE3MDkxNzM2NTIxOTE0OCwgMS4wODgxNDk2MDU3OTYzMTUxLCAxLjA1OTk1NzMyMTIxNDU0NzIsIDEuMDMyNDk1NDU1NDE1LCAxLjAwNTc0NTA4NDM2OTE1NDldLCAiY29uY19pbmYiOiBbMC4wLCAwLjMxMTY4MTEyMTE2NDY5NjE1LCAwLjYyMTczMDIwNDI5MTcxNiwgMC45MzAxNTU3OTUxMjg1MjkxLCAxLjIzNjk2NjM5NDY3NTAyNiwgMS41NDIxNzA0NTk0MTc3Njk2LCAxLjg0NTc3NjQwMTU2MzExNDYsIDIuMTQ3NzkyNTg5MjY5MDY2LCAyLjQ0ODIyNzM0Njg3NTkyMiwgMi43NDcwODg5NTUxMzU2OTgzLCAzLjA0NDM4NTY1MTQ0MDQwMTMsIDMuMzQwMTI1NjMwMDQ5MDM4MywgMy42MzQzMTcwNDIzMTM0OTgzLCAzLjkyNjk2Nzk5NjkwMzIwNDcsIDQuMjE4MDg2NTYwMDI4NjE1LCA0LjUwNzY4MDc1NTY2MzU1NSwgNC43OTU3NTg1NjU3NjYzODksIDUuMDgyMzI3OTMwNDk5OTc5NiwgNS4zNjczOTY3NDg0NTA1OTgsIDUuNjUwOTcyODc2ODQ1NTcyLCA1LjkzMzA2NDEzMTc2OTkwMywgNi4yMTM2NzgyODgzODE2NjMsIDYuNDkyODIzMDgxMTI2MzAzLCA2Ljc3MDUwNjIwMzk0OTg0NSwgNy4wNDY3MzUzMTA1MTA5NDg1LCA3LjMyMTUxODAxNDM5MTg0NCwgNy41OTQ4NjE4ODkzMDgyMSwgNy44NjY3NzQ0NjkzMTc5LCA4LjEzNzI2MzI0OTAyODYwNSwgOC40MDYzMzU2ODM4MDQ0NTksIDguNjczOTk5MTg5OTcxNDYzLCA4Ljk0MDI2MTE0NTAyMTk1NSwgOS4yMDUxMjg4ODc4MTc5MzEsIDkuNDY4NjA5NzE4NzkzMzI2LCA5LjczMDcxMDkwMDE1NTIwNywgOS45OTE0Mzk2NTYwODM5OTQsIDEwLjI1MDgwMzE3MjkzMjUyMSwgMTAuNTA4ODA4NTk5NDI0MTQyLCAxMC43NjU0NjMwNDY4NDk3NjUsIDExLjAyMDc3MzU4OTI2MzgzNCwgMTEuMjc0NzQ3MjYzNjc5MzQsIDExLjIxNTcwOTk0OTA5NzA3MiwgMTEuMTU2OTgxNzY4MjMwMzE1LCAxMS4wOTg1NjExMDIzNzk4NjgsIDExLjA0MDQ0NjM0MTMyMjQzMSwgMTAuOTgyNjM1ODgzMjY2MjIsIDEwLjkyNTEyODEzNDgwNjgyOCwgMTAuODY3OTIxNTEwODgzMjk2LCAxMC44MTEwMTQ0MzQ3MzQ0MjgsIDEwLjc1NDQwNTMzNzg1NTM0LCAxMC42OTgwOTI2NTk5NTQyMTMsIDEwLjY0MjA3NDg0ODkwOTI5OCwgMTAuNTg2MzUwMzYwNzI2MTMsIDEwLjUzMDkxNzY1OTQ5NDk3NSwgMTAuNDc1Nzc1MjE3MzQ4NDk5LCAxMC40MjA5MjE1MTQ0MTk2NSwgMTAuMzY2MzU1MDM4Nzk5NzY0LCAxMC4zMTIwNzQyODY0OTY5MDcsIDEwLjI1ODA3Nzc2MTM5NDQwNCwgMTAuMjA0MzYzOTc1MjA5NjEzLCAxMC4xNTA5MzE0NDc0NTI5MDUsIDEwLjA5Nzc3ODcwNTM4Njg1LCAxMC4wNDQ5MDQyODM5ODU2MjcsIDkuOTkyMzA2NzI1ODk0NjUyLCA5LjkzOTk4NDU4MTM5MDQsIDkuODg3OTM2NDA4MzQwNDUsIDkuODM2MTYwNzcyMTYzNzM1LCA5Ljc4NDY1NjI0NTc5MTAwNywgOS43MzM0MjE0MDk2MjU1MDEsIDkuNjgyNDU0ODUxNTAzNzk1LCA5LjYzMTc1NTE2NjY1NjkwOSwgOS41ODEzMjA5NTc2NzE1NjcsIDkuNTMxMTUwODM0NDUxNjkyLCA5LjQ4MTI0MzQxNDE4MDA4NywgOS40MzE1OTczMjEyODAzMjEsIDkuMzgyMjExMTg3Mzc4ODE1LCA5LjMzMzA4MzY1MTI2NzEyOCwgOS4yODQyMTMzNTg4NjQ0MzMsIDkuMjM1NTk4OTYzMTgwMjA0LCA5LjE4NzIzOTEyNDI3NzA4MywgOS4xMzkxMzI1MDkyMzM5NDcsIDkuMDkxMjc3NzkyMTA5MTc1LCA5LjA0MzY3MzY1MzkwNDA5NCwgOC45OTYzMTg3ODI1MjY2MzQsIDguOTQ5MjExODcyNzU1MTUyLCA4LjkwMjM1MTYyNjIwMjQ2OSwgOC44NTU3MzY3NTEyODAwNjgsIDguODA5MzY1OTYzMTYyNTExLCA4Ljc2MzIzNzk4Mzc1MjAxNSwgOC43MTczNTE1NDE2NDMyMjcsIDguNjcxNzA1MzcyMDg4MTc4LCA4LjYyNjI5ODIxNjk2MTQzNCwgOC41ODExMjg4MjQ3MjU0MDYsIDguNTM2MTk1OTUwMzk1ODYsIDguNDkxNDk4MzU1NTA3NjA3LCA4LjQ0NzAzNDgwODA4MDM1MSwgOC40MDI4MDQwODI1ODQ3NTcsIDguMzU4ODA0OTU5OTA4NjUsIDguMzE1MDM2MjI3MzIzNDI5LCA4LjI3MTQ5NjY3ODQ1MDYyNiwgOC4yMjgxODUxMTMyMjg2NzQsIDguMTg1MTAwMzM3ODc5ODEyLCA4LjE0MjI0MTE2NDg3NzE5MSwgOC4wOTk2MDY0MTI5MTIxNDUsIDguMDU3MTk0OTA2ODYxNjIzLCA4LjAxNTAwNTQ3Nzc1NTgwMiwgNy45NzMwMzY5NjI3NDU4NzUsIDcuOTMxMjg4MjA1MDcxOTg2LCA3Ljg4OTc1ODA1NDAzMTM2MSwgNy44NDg0NDUzNjQ5NDY1ODQsIDcuODA3MzQ4OTk5MTM0MDQ0LCA3Ljc2NjQ2NzgyMzg3MjU1NywgNy43MjU4MDA3MTIzNzIxNDQsIDcuNjg1MzQ2NTQzNzQyOTY3LCA3LjY0NTEwNDIwMjk2NDQ0NSwgNy42MDUwNzI1ODA4NTQ1MSwgNy41NjUyNTA1NzQwMzkwNDQ2LCA3LjUyNTYzNzA4NDkyMTQ2NCwgNy40ODYyMzEwMjE2NTI0Njc1LCA3LjQ0NzAzMTI5ODA5OTk0LCA3LjQwODAzNjgzMzgxOTAyMywgNy4zNjkyNDY1NTQwMjIzMjMsIDcuMzMwNjU5Mzg5NTUwMjk3LCA3LjI5MjI3NDI3Njg0MTc4MywgNy4yNTQwOTAxNTc5MDQ2ODEsIDcuMjE2MTA1OTgwMjg2Nzk2LCA3LjE3ODMyMDY5NzA0NjgyODUsIDcuMTQwNzMzMjY2NzI1NTE1LCA3LjEwMzM0MjY1MzMxNjkzLCA3LjA2NjE0NzgyNjIzOTkyMywgNy4wMjkxNDc3NjAzMDk3MTY2LCA2Ljk5MjM0MTQzNTcwOTY1MywgNi45NTU3Mjc4Mzc5NjMwODIsIDYuOTE5MzA1OTU3OTA1Mzk0LCA2Ljg4MzA3NDc5MTY1NjIxNywgNi44NDcwMzMzNDA1OTE3MzYsIDYuODExMTgwNjExMzE3MTc1LCA2Ljc3NTUxNTYxNTYzOTQxMywgNi43NDAwMzczNzA1Mzk3NDksIDYuNzA0NzQ0ODk4MTQ2ODA3LCA2LjY2OTYzNzIyNTcwOTU4MywgNi42MzQ3MTMzODU1NzA2MywgNi41OTk5NzI0MTUxMzkzOTcsIDYuNTY1NDEzMzU2ODY1NjgzLCA2LjUzMTAzNTI1ODIxMzI2MSwgNi40OTY4MzcxNzE2MzM2MDYsIDYuNDYyODE4MTU0NTM5Nzk2LCA2LjQyODk3NzI2OTI4MDUxNSwgNi4zOTUzMTM1ODMxMTQyMjI1LCA2LjM2MTgyNjE2ODE4MzQzNywgNi4zMjg1MTQxMDE0ODkxNjUsIDYuMjk1Mzc2NDY0ODY1NDU3LCA2LjI2MjQxMjM0NDk1NDEwOSwgNi4yMjk2MjA4MzMxNzk0NzcsIDYuMTk3MDAxMDI1NzIzNDQzLCA2LjE2NDU1MjAyMzUwMDQ5OSwgNi4xMzIyNzI5MzIxMzI5NjQsIDYuMTAwMTYyODYxOTI2MzQyLCA2LjA2ODIyMDkyNzg0NDc4OSwgNi4wMzY0NDYyNDk0ODY3MjUsIDYuMDA0ODM3OTUxMDYwNTY0LCA1Ljk3MzM5NTE2MTM2MDU4NCwgNS45NDIxMTcwMTM3NDI4OTksIDUuOTExMDAyNjQ2MTAxNTg4LCA1Ljg4MDA1MTIwMDg0NDkyMSwgNS44NDkyNjE4MjQ4NzE3MzEsIDUuODE4NjMzNjY5NTQ3ODkxLCA1Ljc4ODE2NTg5MDY4MjkzMywgNS43NTc4NTc2NDg1MDY3NzA1LCA1LjcyNzcwODEwNzY0NjU2MywgNS42OTc3MTY0MzcxMDM2ODEsIDUuNjY3ODgxODEwMjMwODA0LCA1LjYzODIwMzQwNDcwOTE0NCwgNS42MDg2ODA0MDI1MjU3NjcsIDUuNTc5MzExOTg5OTUxMDU3LCA1LjU1MDA5NzM1NzUxNjI4MywgNS41MjEwMzU2OTk5OTEyODksIDUuNDkyMTI2MjE2MzYyMzAxLCA1LjQ2MzM2ODEwOTgwOTg0NSwgNS40MzQ3NjA1ODc2ODY3ODgsIDUuNDA2MzAyODYxNDk2NDkyLCA1LjM3Nzk5NDE0Njg3MTA3NiwgNS4zNDk4MzM2NjM1NDk4MDIsIDUuMzIxODIwNjM1MzU3NTY0NiwgNS4yOTM5NTQyOTAxODM1MDEsIDUuMjY2MjMzODU5OTU5NzA5LCA1LjIzODY1ODU4MDY0MDA3NiwgNS4yMTEyMjc2OTIxNzkyMTgsIDUuMTgzOTQwNDM4NTExNTM3LCA1LjE1Njc5NjA2NzUzMDM3NSwgNS4xMjk3OTM4MzEwNjcyOSwgNS4xMDI5MzI5ODQ4NzE0MjgsIDUuMDc2MjEyNzg4NTg5MDE3LCA1LjA0OTYzMjUwNTc0Mjk1NSwgNS4wMjMxOTE0MDM3MTI1MTIsIDQuOTk2ODg4NzUzNzEzMTQyLCA0Ljk3MDcyMzgzMDc3NjM4NiwgNC45NDQ2OTU5MTM3Mjk5MDIsIDQuOTE4ODA0Mjg1MTc3NTc3LCA0Ljg5MzA0ODIzMTQ3OTc1OSwgNC44Njc0MjcwNDI3MzM1OTEsIDQuODQxOTQwMDEyNzUzNDMyLCA0LjgxNjU4NjQzOTA1MTQwNywgNC43OTEzNjU2MjI4MTgwMzYsIDQuNzY2Mjc2ODY4OTAyOTczLCA0Ljc0MTMxOTQ4NTc5NTg1MywgNC43MTY0OTI3ODU2MDcyMiwgNC42OTE3OTYwODQwNDk1ODEsIDQuNjY3MjI4NzAwNDE4NTM2LCA0LjY0Mjc4OTk1NzU3NDAxNiwgNC42MTg0NzkxODE5MjE2MjgsIDQuNTk0Mjk1NzAzMzk0MDc2LCA0LjU3MDIzODg1NTQzMjcwMiwgNC41NDYzMDc5NzQ5NjkxMTQsIDQuNTIyNTAyNDAyNDA2OTAxLCA0LjQ5ODgyMTQ4MTYwMzQ2NSwgNC40NzUyNjQ1NTk4NTE5MjQsIDQuNDUxODMwOTg3ODYzMTMyLCA0LjQyODUyMDExOTc0Nzc3OTUsIDQuNDA1MzMxMzEyOTk4NTg2LCA0LjM4MjI2MzkyODQ3MjU5OSwgNC4zNTkzMTczMzAzNzM1NzEsIDQuMzM2NDkwODg2MjM0NDM5LCA0LjMxMzc4Mzk2Njg5OTg5MywgNC4yOTExOTU5NDY1MDkwMywgNC4yNjg3MjYyMDI0NzgxMSwgNC4yNDYzNzQxMTU0ODMzODcsIDQuMjI0MTM5MDY5NDQ0MDQ5LCA0LjIwMjAyMDQ1MTUwNTIzNSwgNC4xODAwMTc2NTIwMjExMzUsIDQuMTU4MTMwMDY0NTM4MTk3LCA0LjEzNjM1NzA4NTc3ODQwMiwgNC4xMTQ2OTgxMTU2MjI2NDksIDQuMDkzMTUyNTU3MDk0MTk5LCA0LjA3MTcxOTgxNjM0MjIzMiwgNC4wNTAzOTkzMDI2MjU0NzUsIDQuMDI5MTkwNDI4Mjk1OTE3LCA0LjAwODA5MjYwODc4MjYxOCwgMy45ODcxMDUyNjI1NzU1OTE2LCAzLjk2NjIyNzgxMTIwOTc3ODQsIDMuOTQ1NDU5Njc5MjQ5MTA2XSwgIm5fZ3JpZCI6IDk3LCAibl9ncmlkX2luZiI6IDI0MSwgInRfY21heCI6IDIuMCwgInRfZW5kIjogMTIuMH0=").decode("utf-8"))

CHECK_RESULTS = []


class _Todo(Exception):
    """Raised when a part has not been attempted yet."""


def _need(name):
    g = globals()
    if name not in g:
        raise _Todo("`" + name + "` is not defined yet. Run the starter cell for this part.")
    val = g[name]
    if val is None:
        raise _Todo("`" + name + "` is still None. Fill in the # YOUR CODE blank.")
    return val


def _as_scalar(val):
    try:
        arr = _np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.size != 1:
        return None
    return float(arr.reshape(-1)[0])


def _as_array(val):
    try:
        return _np.asarray(val, dtype=float)
    except Exception:
        return None


def _num_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_scalar(a)
    if a is None:
        return False
    return bool(_np.isclose(a, float(b), rtol=rtol, atol=atol, equal_nan=True))


def _arr_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_array(a)
    if a is None:
        return False
    b = _np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(_np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True))


def _cols_of(obj):
    try:
        return [str(c) for c in obj.columns]
    except Exception:
        return None


def check():
    """Test every part and print PASS, FAIL or TODO for each one."""
    global CHECK_RESULTS
    names = sorted(n for n in globals() if n.startswith("_part_"))
    CHECK_RESULTS = []
    for name in names:
        label = name[len("_part_"):]
        try:
            problem = globals()[name]()
            status = "FAIL" if problem else "PASS"
            msg = problem or ""
        except _Todo as exc:
            status, msg = "TODO", str(exc)
        except Exception as exc:
            status = "FAIL"
            msg = "could not check this part, it raised %s: %s" % (type(exc).__name__, exc)
        CHECK_RESULTS.append((label, status, msg))
    _print_results()


def _print_results():
    bar = "=" * 70
    print(bar)
    print("Problem %s  |  %s" % (PROBLEM_ID, TOPIC))
    print(bar)
    for label, status, msg in CHECK_RESULTS:
        print("Part %s: %s" % (label, status))
        if msg:
            for line in _tw.wrap(msg, width=64):
                print("    " + line)
    n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
    print("-" * 70)
    print("Result: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    if n_pass == len(CHECK_RESULTS):
        print("All parts pass. Run the last cell to build your tutor block.")
    print(bar)


def send_to_tutor():
    """Build the block you copy to your tutor, along with your code."""
    lines = []
    lines.append("===== SEND TO TUTOR =====")
    lines.append("Problem ID : %s" % PROBLEM_ID)
    lines.append("Topic      : %s" % TOPIC)
    lines.append("Date       : %s" % _dt.date.today().isoformat())
    lines.append("Check results:")
    if not CHECK_RESULTS:
        lines.append("  check() has not been run yet. Run the check cell above first.")
    else:
        for label, status, msg in CHECK_RESULTS:
            lines.append("  Part %s: %s" % (label, status))
            if msg:
                for line in _tw.wrap(msg, width=58):
                    lines.append("      " + line)
        n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        lines.append("Summary: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    lines.append("=========================")
    lines.append("Paste your code cells below this block.")
    return "\n".join(lines)



def _part_a():
    rhs = _need("rhs")
    sol = _need("sol")
    conc_num = _need("conc_num")

    if not callable(rhs):
        return "`rhs` should be a function."
    try:
        probe = rhs(0.0, [100.0], 4.2, 40.0)
    except Exception as exc:
        return "`rhs` raised %s when called as rhs(t, y, cl, v)." % type(exc).__name__
    if probe is None:
        return "`rhs` returns None. Replace the # YOUR CODE line."
    arr = _as_array(probe)
    if arr is None or arr.ndim == 0:
        return ("`rhs` should return a SEQUENCE of derivatives, one per state, so a list "
                "like [-(cl/v) * y[0]] rather than a bare number.")
    if not _np.isclose(float(_np.ravel(arr)[0]), -(4.2 / 40.0) * 100.0):
        if _np.isclose(float(_np.ravel(arr)[0]), (4.2 / 40.0) * 100.0):
            return "`rhs` has the wrong sign. The amount is falling, so the derivative is negative."
        return "`rhs` does not compute -(cl / v) * y[0]."

    if not hasattr(sol, "success"):
        return "`sol` should be the object solve_ivp returns."
    if not sol.success:
        return "`sol` reports success = False, so the integration failed."
    if _np.asarray(sol.t).size != _EXP["n_grid"]:
        return ("`sol` reported at %d times but T_GRID has %d. Pass t_eval=T_GRID so the "
                "output lands on the grid you want." % (_np.asarray(sol.t).size, _EXP["n_grid"]))

    arr = _as_array(conc_num)
    if arr is None:
        return "`conc_num` should be a numeric array."
    if arr.ndim == 2:
        return ("`conc_num` is two dimensional, so it is still sol.y. Take the first state "
                "with sol.y[0].")
    if arr.shape != (_EXP["n_grid"],):
        return "`conc_num` should have one value per time in T_GRID, so shape (%d,)." % _EXP["n_grid"]
    if _arr_close(arr, _EXP["amount"], rtol=1e-4):
        return ("`conc_num` is still an AMOUNT in mg. The state is mass, so divide by the "
                "volume to get a concentration in mg/L.")
    if not _arr_close(arr, _EXP["conc_exact"], rtol=1e-5, atol=1e-8):
        if _arr_close(arr, _EXP["conc_exact"], rtol=2e-2, atol=1e-3):
            return ("`conc_num` is close but only to about three digits, which is the "
                    "signature of solve_ivp running on its default tolerances. Pass "
                    "rtol=1e-8, atol=1e-10.")
        return ("`conc_num` does not match the expected profile. Check the starting amount "
                "is the full dose and that args passes CL then V.")
    return None


def _part_b():
    conc_exact = _need("conc_exact")
    max_abs_err = _need("max_abs_err")

    arr = _as_array(conc_exact)
    if arr is None:
        return "`conc_exact` should be a numeric array."
    if arr.shape != (_EXP["n_grid"],):
        return "`conc_exact` should have one value per time in T_GRID."
    if _arr_close(arr, _EXP["amount"], rtol=1e-6):
        return ("`conc_exact` is an amount in mg. Divide the dose by the volume before the "
                "exponential, so it is a concentration.")
    if not _arr_close(arr, _EXP["conc_exact"], rtol=1e-6):
        return ("`conc_exact` is not (dose / V) * exp(-(CL/V) * t). Check the rate constant "
                "is CL divided by V.")

    got = _as_scalar(max_abs_err)
    if got is None:
        return "`max_abs_err` should be a single number. np.max reduces the whole array."
    if got < 0:
        return "`max_abs_err` is negative, so the absolute value was not taken."
    if 1e-4 < got < 5e-2:
        return ("`max_abs_err` is %.4g. That is the signature of solve_ivp running on its "
                "default tolerances, which only ask for about three correct digits. Pass "
                "rtol=1e-8, atol=1e-10 to the solve in part a." % got)
    if got > 1e-4:
        return ("`max_abs_err` is %.4g, which is far too large. The two curves should agree "
                "to roughly 1e-8. Suspect the volume division or the starting amount." % got)
    return None


def _part_c():
    rhs_inf = _need("rhs_inf")
    sol_inf = _need("sol_inf")
    conc_inf = _need("conc_inf")
    cmax_inf = _need("cmax_inf")
    t_cmax = _need("t_cmax")

    if not callable(rhs_inf):
        return "`rhs_inf` should be a function."
    try:
        on = rhs_inf(0.5, [0.0], 4.2, 40.0, 250.0, 2.0)
        off = rhs_inf(5.0, [100.0], 4.2, 40.0, 250.0, 2.0)
    except Exception as exc:
        return ("`rhs_inf` raised %s when called as rhs_inf(t, y, cl, v, rate, t_off)."
                % type(exc).__name__)
    if on is None or off is None:
        return "`rhs_inf` returns None. Replace the # YOUR CODE line."
    on_v = float(_np.ravel(_as_array(on))[0])
    off_v = float(_np.ravel(_as_array(off))[0])
    if not _np.isclose(on_v, 250.0):
        return ("During the infusion, with an empty compartment, the derivative should be "
                "the input rate alone. Got %.4f rather than 250." % on_v)
    if not _np.isclose(off_v, -(4.2 / 40.0) * 100.0):
        if _np.isclose(off_v, 250.0 - (4.2 / 40.0) * 100.0):
            return ("After t_off the input should be zero, but the rate is still being "
                    "added. Check the condition is t < t_off.")
        return "After the infusion stops the derivative should be elimination only."

    if not hasattr(sol_inf, "success"):
        return "`sol_inf` should be the object solve_ivp returns."
    if not sol_inf.success:
        return "`sol_inf` reports success = False."
    if _np.asarray(sol_inf.t).size != _EXP["n_grid_inf"]:
        return ("`sol_inf` reported at %d times but T_GRID_INF has %d. Pass t_eval."
                % (_np.asarray(sol_inf.t).size, _EXP["n_grid_inf"]))

    arr = _as_array(conc_inf)
    if arr is None or arr.shape != (_EXP["n_grid_inf"],):
        return "`conc_inf` should have one value per time in T_GRID_INF."
    if _arr_close(arr, _EXP["amount_inf"], rtol=1e-3):
        return "`conc_inf` is an amount in mg. Divide by the volume."
    if not _arr_close(arr, _EXP["conc_inf"], rtol=1e-4, atol=1e-7):
        return ("`conc_inf` does not match the expected infusion profile. If it keeps "
                "climbing past 2 hours, the solver stepped over the switch off: pass "
                "max_step=0.05.")

    got = _as_scalar(cmax_inf)
    if got is None:
        return "`cmax_inf` should be a single number."
    if _num_close(got, _EXP["cmax_amount"], rtol=1e-3):
        return "`cmax_inf` is an amount in mg, not a concentration."
    if not _num_close(got, _EXP["cmax_inf"], rtol=1e-4):
        return "`cmax_inf` is not the peak of conc_inf."

    got = _as_scalar(t_cmax)
    if got is None:
        return "`t_cmax` should be a single number."
    if _num_close(got, float(_np.argmax(_np.asarray(_EXP["conc_inf"])))):
        return ("`t_cmax` is the index of the peak, not the time. Use it to look up the "
                "matching entry of sol_inf.t.")
    if not _num_close(got, _EXP["t_cmax"], rtol=1e-6, atol=1e-6):
        if got > _EXP["t_cmax"]:
            return ("`t_cmax` is after the infusion ended, which cannot happen: once the "
                    "input stops the concentration can only fall. Either the condition in "
                    "rhs_inf is not switching the rate off, or the solver stepped over the "
                    "switch off, which max_step=0.05 prevents.")
        return "`t_cmax` should be the moment the infusion stops, which is when the peak occurs."
    return None


check()


## Send to tutor

In [ ]:
# Run this last. Copy everything it prints, then paste your own code under it.
print(send_to_tutor())
